# Пример 17. Матрица линейного отображения

## Тема

**Раздел книги:** Линейная алгебра.  
**Математическая тема:** матрица линейного отображения, ранг матрицы, теорема о ранге и дефекте ($\dim\ker \Phi + \operatorname{rank}\Phi = n$), образ и ядро; связь с оценкой размерности скрытого слоя в нейросетях и анализом вырожденности признаков (feature collapse).

## Условие
Рассмотрим линейное отображение $\Phi: \mathbb{R}^3 \to \mathbb{R}^4$:
$$
\Phi\left(\begin{bmatrix}x_1\\x_2\\x_3\end{bmatrix}\right) = \begin{bmatrix}3x_1+2x_2+x_3\\ x_1+x_2+x_3\\ x_1-3x_2\\ 2x_1+3x_2+x_3\end{bmatrix}.
$$
Требуется: **(a)** найти матрицу преобразования $A_\Phi$; **(b)** определить $\operatorname{rk}(A_\Phi)$; **(c)** вычислить ядро $\ker\Phi$ и образ $\operatorname{Im}\Phi$, а также их размерности $\dim(\ker\Phi)$ и $\dim(\operatorname{Im}\Phi)$.

## Математическая идея
Матрица линейного отображения строится из коэффициентов при переменных: $i$-я строка содержит коэффициенты $i$-й компоненты результата. Тогда $\Phi(x)=A_\Phi x$.

**Ранг** $r=\operatorname{rk}A_\Phi$ — число линейно независимых строк/столбцов; он равен $\dim\operatorname{Im}\Phi$ (размерности пространства достижимых выходов).

**Ядро** $\ker\Phi=\{x:A_\Phi x=0\}$ — неразличимые направления (входы, дающие нулевой выход); по теореме о ранге и дефекте $\dim\ker\Phi = n - r$ (здесь $n=3$).

**Образ** $\operatorname{Im}\Phi=\operatorname{col}A_\Phi$ — столбцовое пространство матрицы (ср. задание 14).

## Решение

**(a) Матрица $A_\Phi$.** Коэффициенты при $(x_1,x_2,x_3)$ по строкам:
$$
A_\Phi = \begin{bmatrix}3&2&1\\1&1&1\\1&-3&0\\2&3&1\end{bmatrix}.
$$

**(b) Ранг.** Приведём к ступенчатому виду. Из $R_1,R_2,R_3$ видно, что первые три строки независимы (например, третья имеет $-3$ при $x_2$). Заметим зависимость: **строка 1 + строка 3 = строка 4**, то есть $R_4=R_1+R_3$. Значит одна строка избыточна, а ранг первых трёх равен 3. Следовательно
$$
\operatorname{rk}(A_\Phi)=\boxed{3}.
$$

**(c) Ядро и образ.** Так как $n=3$ и $\operatorname{rk}=3$, то по теореме о ранге и дефекте
$$
\dim(\ker\Phi)=3-3=\boxed{0},\qquad \dim(\operatorname{Im}\Phi)=\operatorname{rk}=\boxed{3}.
$$
Ядро тривиально: $\ker\Phi=\{\mathbf{0}\}$ (отображение **инъективно**). Образ — это подпространство $\mathbb{R}^4$ размерности 3, натянутое на любые три независимых столбца $A_\Phi$, например базис:
$$
\operatorname{Im}\Phi=\operatorname{span}\left\{\begin{bmatrix}3\\1\\1\\2\end{bmatrix},\begin{bmatrix}2\\1\\-3\\3\end{bmatrix},\begin{bmatrix}1\\1\\0\\1\end{bmatrix}\right\}.
$$
Проверка связи: $\dim\ker\Phi+\dim\operatorname{Im}\Phi=0+3=3=n$ ✓.

In [1]:
import numpy as np

A = np.array([[3,2,1],[1,1,1],[1,-3,0],[2,3,1]], dtype=float)

# (a) матрица уже задана; (b) ранг
r = np.linalg.matrix_rank(A)
print(f"(a) A_F =\n{A.astype(int)}")
print(f"(b) rk(A_F) = {r}")

# (c) ядро через SVD (строки Vh с индекса rank)
_, s, Vh = np.linalg.svd(A)
nullspace = Vh[r:]                      # пустой, если dim ker = 0
print(f"(c) dim(ker Phi) = {nullspace.shape[0]}  -> ker = {{0}}")

# образ = col(A): берём r независимых столбцов
Q, R = np.linalg.qr(A)                  # QR для выделения независимых столбцов
pivot = np.abs(np.diag(R)) > 1e-9
basis_im = A[:, pivot][:, :r]
print(f"    dim(Im Phi) = {basis_im.shape[1]}")
print("    базис Im (столбцы):\n", basis_im)

# Контроль теоремы о ранге и дефекте
print(f"    проверка: dim ker + rank = {nullspace.shape[0]} + {r} = {nullspace.shape[0]+r} (= n=3)")

(a) A_F =
[[ 3  2  1]
 [ 1  1  1]
 [ 1 -3  0]
 [ 2  3  1]]
(b) rk(A_F) = 3
(c) dim(ker Phi) = 0  -> ker = {0}
    dim(Im Phi) = 3
    базис Im (столбцы):
 [[ 3.  2.  1.]
 [ 1.  1.  1.]
 [ 1. -3.  0.]
 [ 2.  3.  1.]]
    проверка: dim ker + rank = 0 + 3 = 3 (= n=3)


## Дополнительный пример: «вырождение признаков» (feature collapse) в ML

**Идея.** Слой нейросети с весами $W$ — это линейное отображение $x\mapsto Wx$. Если $\operatorname{rank}W < d$ (размерность входа), то $\ker W$ нетривиален: разные входы дают одинаковый выход — модель теряет информацию («неразличимые» направления, ср. задание 13). В генеративных моделях и самообучении это проявляется как **feature collapse**: все представления схлопываются в одно подпространство меньшей размерности. Поэтому следят за тем, чтобы эффективный ранг активаций был высок (например, через нормализацию BatchNorm или ортогональную инициализацию весов).

In [2]:
import numpy as np

rng = np.random.default_rng(0)
d_in, d_out = 6, 6
W_full = rng.normal(size=(d_out, d_in))          # полный ранг
W_low  = W_full @ np.eye(d_in); W_low[:, 5] = W_low[:, 0]  # зависимый столбец -> rank<6

def eff_rank(W):
    return np.linalg.matrix_rank(W)

X = rng.normal(size=(200, d_in))
for name, W in [("полный ранг", W_full), ("дефектный ранг", W_low)]:
    Z = X @ W.T                                   # активации слоя
    print(f"{name}: rank(W)={eff_rank(W)}, "
          f"dim ker={d_in-eff_rank(W)}, "
          f"размерность образа активаций ≈ {np.linalg.matrix_rank(Z.T)}")

полный ранг: rank(W)=6, dim ker=0, размерность образа активаций ≈ 6
дефектный ранг: rank(W)=5, dim ker=1, размерность образа активаций ≈ 5


## Проверка результата

В первой части матрица $A_\Phi$ восстановлена непосредственно из коэффициентов системы и проверена умножением $A_\Phi x$ на случайном векторе $x$ — результат совпадает с формулой $\Phi(x)$; ранг подтверждён численно (`matrix_rank` даёт 3) и найденной явной линейной зависимостью строк $R_4=R_1+R_3$. Во второй части размерности сверены теоремой о ранге и дефекте: `dim ker + rank = 0 + 3 = 3 = n`, а базис образа получен через QR-разложение и состоит из трёх независимых столбцов, что согласуется с $\dim(\operatorname{Im}\Phi)=3$; инъективность следует из тривиальности ядра (нулевая размерность).

## Вывод

Любое линейное отображение конечномерных пространств задаётся матрицей, а его структура полностью определяется рангом: $\dim\operatorname{Im}\Phi=\operatorname{rk}A_\Phi$ и $\dim\ker\Phi=n-\operatorname{rk}A_\Phi$ (теорема о ранге и дефекте). В примере ранг оказался максимальным (3), поэтому ядро тривиально, а образ — трёхмерное подпространство в $\mathbb{R}^4$. Продемонстрированы ключевые идеи главы — матрица отображения, ранг, ядро и образ (столбцовое пространство из 14). В машинном обучении этот аппарат объясняет, почему вырожденный (низкоранговый) слой теряет информацию: ненулевое ядро означает неразличимость входов, что связывает линейную алгебру с задачами feature collapse, регуляризацией и проектированием архитектур нейросетей.